In [108]:
!pip install ortools

In [109]:
"""algorithm_comparison.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1V6U6-l3cbofgAbOLFzOxgqv58kbhRi15

# BENCHMARK: SO SANH THUAT TOAN XEP LICH BENH VIEN

## Cac thuat toan duoc so sanh:

| # | Thuật toán | Mo ta | Trạng thái |
|---|-----------|-------|-----------|
| 1 | Greedy | Ưu tiên chọn người ít công nhất | ✅ Implemented |
| 2 | Enhanced Greedy | Greedy + fatigue awareness | ✅ Implemented |
| 3 | Beam Search | Tim kiem chum voi K state tot nhat | ✅ Implemented |
| 4 | Simulated Annealing | Toi uu hoa bang SA, thoat local optima | ✅ Implemented |
| 5 | Hill Climbing | Leo đồi co ban | ✅ Implemented |
| 6 | Tabu Search | Tim kiem tabu tranh lap lai | ✅ Implemented |
| 7 | Genetic | Thuat toan di truyen | ✅ Implemented |
| 8 | Hybrid | Ket hop Greedy + Local Search | ✅ Implemented |

### Dac diem:
- **Data models**: Dua theo app/models.py
- **Scoring**: Dua theo app/scoring.py
- **Constraints**: Dua theo app/constraints.py
- **Co the import** tu app hoac chay standalone
"""

'algorithm_comparison.ipynb\n\nAutomatically generated by Colab.\n\nOriginal file is located at\n    https://colab.research.google.com/drive/1V6U6-l3cbofgAbOLFzOxgqv58kbhRi15\n\n# BENCHMARK: SO SANH THUAT TOAN XEP LICH BENH VIEN\n\n## Cac thuat toan duoc so sanh:\n\n| # | Thuật toán | Mo ta | Trạng thái |\n|---|-----------|-------|-----------|\n| 1 | Greedy | Ưu tiên chọn người ít công nhất | ✅ Implemented |\n| 2 | Enhanced Greedy | Greedy + fatigue awareness | ✅ Implemented |\n| 3 | Beam Search | Tim kiem chum voi K state tot nhat | ✅ Implemented |\n| 4 | Simulated Annealing | Toi uu hoa bang SA, thoat local optima | ✅ Implemented |\n| 5 | Hill Climbing | Leo đồi co ban | ✅ Implemented |\n| 6 | Tabu Search | Tim kiem tabu tranh lap lai | ✅ Implemented |\n| 7 | Genetic | Thuat toan di truyen | ✅ Implemented |\n| 8 | Hybrid | Ket hop Greedy + Local Search | ✅ Implemented |\n\n### Dac diem:\n- **Data models**: Dua theo app/models.py\n- **Scoring**: Dua theo app/scoring.py  \n- **Constrai

In [110]:
# ============================================================
# CELL 1: IMPORTS + CORE DATA MODELS
# ============================================================
import time, json, math, random, copy, numpy as np
from datetime import date, timedelta
from typing import Dict, Tuple, Set, List, Optional
import sys
import os

In [111]:
# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

============================================================
SECTION 1: CORE DATA MODELS (Standalone - match app/models.py)
============================================================

In [112]:
class Staff:
    def __init__(self, id: int, name: str, specialty: str = "GENERAL",
                 specialty_id: int = 1, max_shifts: int = 5, is_active: bool = True):
        self.id = id
        self.name = name
        self.specialty = specialty
        self.specialty_id = specialty_id  # NEW: map to shift_requirement.specialty_id
        self.max_shifts = max_shifts
        self.is_active = is_active
    def __repr__(self):
        return f"Staff({self.id}, {self.name}, spec={self.specialty_id})"

In [113]:
class LeaveRequest:
    def __init__(self, staff_id: int, work_date: date, reason: str = "", status: str = "PENDING"):
        self.staff_id = staff_id
        self.date = work_date
        self.reason = reason
        self.status = status  # APPROVED | PENDING

In [114]:
import datetime
from datetime import date, timedelta
from typing import Set, Tuple

# Giả lập danh sách ngày lễ
HOLIDAYS = {datetime.date(2024, 6, 10), datetime.date(2024, 6, 11)}

def calculate_compensation_date(truc_date: date, staff_id: int, leave_index: Set[Tuple[int, date]]) -> date:
    """Tính ngày nghỉ bù sát theo đặc tả gốc (Gồm cả cuối tuần sau)."""
    weekday = truc_date.weekday()  # 0=Mon, 6=Sun

    # 1. Xác định ngày 'dự kiến'
    if weekday == 4 or weekday == 5:  # Thứ 6 hoặc Thứ 7
        # Nghỉ bù vào tuần sau, bắt đầu xét từ Thứ 3 tuần sau
        target_date = truc_date + timedelta(days=(8 - weekday) + 1)
    elif weekday == 6:  # Chủ Nhật
        # Nghỉ vào Thứ 2 tuần sau
        target_date = truc_date + timedelta(days=1)
    else:  # Thứ 2 đến Thứ 5
        # Nghỉ ngay hôm sau
        target_date = truc_date + timedelta(days=1)

    # 2. Logic Tịnh tiến & Loại trừ (Thứ 2, Thứ 6 cho ca trực T6/T7)
    while True:
        is_holiday = target_date in HOLIDAYS
        is_leave = (staff_id, target_date) in leave_index

        # Quy tắc: Trực T6/T7 không nghỉ bù vào T2 hoặc T6 tuần sau
        is_excluded = False
        if (weekday == 4 or weekday == 5) and target_date.weekday() in [0, 4]:
            is_excluded = True

        if is_holiday or is_leave or is_excluded:
            target_date += timedelta(days=1)
        else:
            break

    return target_date

In [115]:
def assign_with_overnight(schedule, comp_days, req_comp_days,
                          staff_id, work_date, shift_type, leave_index=None):
    """Assign shift va tu dong block ngay hom sau neu la L01 (overnight) kem logic tinh tien."""
    if leave_index is None:
        leave_index = set()

    schedule[(staff_id, work_date)] = shift_type
    if shift_type == "L01":
        # Goi ham tinh toan moi voi day du tham so nghiep vu
        comp_date = calculate_compensation_date(work_date, staff_id, leave_index)
        comp_days.add((staff_id, comp_date))
        req_comp_days.add((staff_id, comp_date))

In [116]:
def can_assign_shift(schedule, leave_index, req_comp_days,
                     staff_id, work_date, shift_type):
    """Check co the assign shift cho staff tai date khong."""
    if (staff_id, work_date) in leave_index:
        return False
    if (staff_id, work_date) in req_comp_days:
        return False
    if any(d == work_date for (sid, d) in req_comp_days if sid == staff_id):
        return False
    return True

In [117]:
def check_specialty_match(staff, shift_requirements, work_date, shift_type):
    """Staff phai co specialty phu hop voi bat ky requirement nao cua ca do."""
    if shift_requirements is None:
        return True
    requirements = shift_requirements.get((work_date, shift_type), {})
    if not requirements:
        return True  # Khong co requirement cu the -> match all
    return staff.specialty_id in requirements

In [118]:
def get_specialty_compatible_candidates(schedule, staff_list, staff_max_shifts,
                                        leave_requests, work_date, shift_type,
                                        shift_requirements, top_n=3):
    """Enhanced candidate filter: specialty + max_shifts + leave."""
    leave_index = build_leave_index(leave_requests)
    workload = build_workload(schedule)
    candidates = []
    for s in staff_list:
        if not s.is_active:
            continue
        if (s.id, work_date) in leave_index:
            continue
        if not check_specialty_match(s, shift_requirements, work_date, shift_type):
            continue
        cnt = workload.get(s.id, 0)
        max_allowed = staff_max_shifts.get(s.id, 5)
        if cnt < max_allowed:
            candidates.append((s.id, cnt))
    candidates.sort(key=lambda x: x[1])
    return [sid for sid, _ in candidates[:top_n]]

In [119]:
# Cac loai ca
SHIFTS = ["L01", "L02", "L03", "L04", "OFF"]
WORK_SHIFTS = ["L01", "L02", "L03", "L04"]
SHIFT_PRIORITY = ["L01", "L02", "L03", "L04"]

In [120]:
# WEIGHT CONFIGURATION (consistent across all algorithms)
WEIGHTS = {
    "fairness":    0.30,
    "fatigue":     0.20,
    "coverage":    0.30,
    "compensation": 0.20,
}

In [121]:
# CONSTRAINT LIMITS
MAX_CONSECUTIVE_WORK_DAYS = 6
MIN_REST_DAYS_BETWEEN_SHIFTS = 1
MIN_GAP_BETWEEN_L01 = 3

In [122]:
print("✅ Cell 1: Data models loaded — Staff, LeaveRequest, calculate_compensation_date defined.")

✅ Cell 1: Data models loaded — Staff, LeaveRequest, calculate_compensation_date defined.


============================================================
CELL 2: CREATE TEST DATA
============================================================

In [123]:
def create_test_data(num_staff=10, num_days=30, max_shifts_per_staff=15, seed=42, num_specialties=2):
    """Tao du lieu test synthetic.

    Args:
        num_nhan_su: So luong nhan su
        num_days: So ngay xep lich
        max_shifts_per_staff: So ca toi da moi nhan su (hard cap)
        seed: Random seed de co the reproduce
        num_specialties: So luong specialty phan biet (default=2)

    Kiem tra tinh khả thi:
        capacity = num_staff * max_shifts_per_staff
        required = 4 shifts/ngay * num_days
    Yêu cầu: capacity > required để algorithms có không gian lựa chọn
    """
    rng = random.Random(seed)
    np.random.seed(seed)

    # Staff với specialty distribution (alternate giữa num_specialties chuyên khoa)
    staff_list = []
    for i in range(num_staff):
        spec_id = (i % num_specialties) + 1
        staff_list.append(Staff(
            id=i + 1,
            name=f"BS.{chr(65 + i)}",
            specialty=f"SPEC_{spec_id}",
            specialty_id=spec_id,
            max_shifts=max_shifts_per_staff,
            is_active=True
        ))

    start_date = date(2024, 6, 1)

    # Tao don nghi phep ngau nhien (20% nhan su nghi/ngay)
    leave_requests = []
    for _ in range(num_staff * num_days // 10):
        sid = rng.randint(1, num_staff)
        day_offset = rng.randint(1, num_days - 2)
        leave_requests.append(LeaveRequest(
            staff_id=sid,
            work_date=start_date + timedelta(days=day_offset),
            reason="Nghi phep",
            status="APPROVED"
        ))

    dates = [start_date + timedelta(days=i) for i in range(num_days)]

    # Yeu cau nhan su toi thieu moi ca/ngay
    min_staff_per_day = {"L01": 1, "L02": 1, "L03": 1, "L04": 1}

    # NEW: Shift requirements per (date, shift_type, specialty)
    # Moi ca can it nhat 1 staff tu moi specialty
    shift_requirements = {}
    for d in dates:
        for shift_type in WORK_SHIFTS:
            shift_requirements[(d, shift_type)] = {
                spec_id: 1 for spec_id in range(1, num_specialties + 1)
            }

    required_count = sum(min_staff_per_day.values()) * num_days
    # FIX: capacity = staff * max_shifts (tổng số ca có thể assign)
    total_capacity = num_staff * max_shifts_per_staff
    # FIX: feasible_ratio cho biết bộ dữ liệu có đủ capacity không
    feasible_ratio = round(total_capacity / required_count, 2) if required_count > 0 else 0

    return {
        "staff_list": staff_list,
        "start_date": start_date,
        "num_days": num_days,
        "leave_requests": leave_requests,
        "dates": dates,
        "min_staff_per_day": min_staff_per_day,
        "shift_requirements": shift_requirements,
        "required_count": required_count,
        "total_capacity": total_capacity,
        "feasible_ratio": feasible_ratio,
        "max_shifts_per_staff": max_shifts_per_staff,
        "num_specialties": num_specialties,
    }

In [124]:
# Instantiate test data
# FIXED: capacity = 10 * 15 = 150 > required = 120
# Các thuật toán giờ có đủ space để assign khác nhau
data = create_test_data(num_staff=10, num_days=30, max_shifts_per_staff=15, seed=42)
staff_list = data["staff_list"]
start_date = data["start_date"]
num_days = data["num_days"]
leave_requests = data["leave_requests"]
dates = data["dates"]
min_staff_per_day = data["min_staff_per_day"]
required_count = data["required_count"]
total_capacity = data["total_capacity"]
max_shifts_per_staff = data["max_shifts_per_staff"]
shift_requirements = data["shift_requirements"]  # NEW: expose to algorithms

In [125]:
staff_max_shifts = {s.id: s.max_shifts for s in staff_list}

In [126]:
print(f"✅ Cell 2: Test data created — {len(staff_list)} nhan su, {num_days} ngay, {len(leave_requests)} don nghi phep")
print(f"   Required: {required_count} | Capacity: {total_capacity} | Feasible ratio: {data['feasible_ratio']:.2f}")
print(f"   max_shifts/nhan_su: {max_shifts_per_staff} | capacity > required: {total_capacity >= required_count}")
print(f"   Staff IDs: {[s.id for s in staff_list]}")
print(f"   Leave requests: {[(lr.staff_id, lr.date) for lr in leave_requests]}")

✅ Cell 2: Test data created — 10 nhan su, 30 ngay, 30 don nghi phep
   Required: 120 | Capacity: 150 | Feasible ratio: 1.25
   max_shifts/nhan_su: 15 | capacity > required: True
   Staff IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
   Leave requests: [(2, datetime.date(2024, 6, 2)), (5, datetime.date(2024, 6, 9)), (4, datetime.date(2024, 6, 6)), (2, datetime.date(2024, 6, 23)), (9, datetime.date(2024, 6, 4)), (10, datetime.date(2024, 6, 15)), (1, datetime.date(2024, 6, 2)), (2, datetime.date(2024, 6, 8)), (4, datetime.date(2024, 6, 18)), (10, datetime.date(2024, 6, 2)), (9, datetime.date(2024, 6, 8)), (9, datetime.date(2024, 6, 15)), (4, datetime.date(2024, 6, 16)), (10, datetime.date(2024, 6, 10)), (1, datetime.date(2024, 6, 26)), (3, datetime.date(2024, 6, 24)), (7, datetime.date(2024, 6, 12)), (5, datetime.date(2024, 6, 6)), (4, datetime.date(2024, 6, 26)), (6, datetime.date(2024, 6, 5)), (2, datetime.date(2024, 6, 14)), (2, datetime.date(2024, 6, 13)), (6, datetime.date(2024, 6, 21)), (5, 

============================================================
CELL 3: UTILITY FUNCTIONS (Helper builders)
============================================================

In [127]:
def build_leave_index(leave_requests: List[LeaveRequest]) -> Set[Tuple[int, date]]:
    """Build a fast lookup set for leave conflicts."""
    return {(lr.staff_id, lr.date) for lr in leave_requests if lr.status == "APPROVED"}

In [128]:
def build_workload(schedule: Dict[Tuple[int, date], str]) -> Dict[int, int]:
    """Count current shifts per staff (excluding OFF)."""
    counts: Dict[int, int] = {}
    for (sid, _), shift in schedule.items():
        if shift != "OFF":
            counts[sid] = counts.get(sid, 0) + 1
    return counts

In [129]:
print("✅ Cell 3: Utility functions loaded — build_leave_index, build_workload")

✅ Cell 3: Utility functions loaded — build_leave_index, build_workload


============================================================
CELL 4: FEASIBILITY CHECK (10 constraints)
============================================================

In [130]:
def feasibility_check(
    schedule: Dict[Tuple[int, date], str],
    staff_list: List[Staff],
    dates: List[date],
    leave_requests: List[LeaveRequest],
    min_staff_per_day: Dict[str, int],
    staff_max_shifts: Dict[int, int],
    required_comp_days: Set[Tuple[int, date]]
) -> Tuple[bool, List[str]]:
    """
    Validate all HARD constraints. Returns (pass, violations).

    Checks performed (10 total):
    1. UNDERSTAFFED: Required shift-slots not filled
    2. OVERSHIFT: Staff exceeds max shifts
    3. LEAVE_CONFLICT: Staff assigned on approved leave day
    4. COMP_CONFLICT: Staff assigned on required compensation day
    5. CONSECUTIVE_WORK: Too many consecutive work days (>6)
    6. NO_REST_GAP: Less than min rest days between work days
    7. DOUBLE_COMP: Same compensation day assigned to multiple staff
    8. L01_GAP: L01 shifts too close together for same staff
    9. STAFF_UNAVAILABLE: Staff marked inactive assigned to work
    10. EMPTY_SHIFT_SLOT: No staff assigned to a required shift slot
    """
    violations: List[str] = []
    sorted_dates = sorted(dates)
    leave_index = build_leave_index(leave_requests)
    workload = build_workload(schedule)

    # Build staff availability lookup
    active_staff_ids = {s.id for s in staff_list if s.is_active}
    inactive_staff_ids = {s.id for s in staff_list if not s.is_active}

    # 1. UNDERSTAFFED
    for d in sorted_dates:
        for shift_type, min_req in min_staff_per_day.items():
            filled = sum(
                1 for (sid, date_key), s in schedule.items()
                if date_key == d and s == shift_type
            )
            if filled < min_req:
                violations.append(
                    f"UNDERSTAFFED: {d} {shift_type} needs {min_req}, got {filled}"
                )

    # 2. OVERSHIFT
    for sid, cnt in workload.items():
        max_allowed = staff_max_shifts.get(sid, 5)
        if cnt > max_allowed:
            violations.append(
                f"OVERSHIFT: staff {sid} has {cnt} shifts > max {max_allowed}"
            )

    # 3. LEAVE_CONFLICT
    for (sid, d), s in schedule.items():
        if s != "OFF" and (sid, d) in leave_index:
            violations.append(f"LEAVE_CONFLICT: staff {sid} assigned on leave {d}")

    # 4. COMP_CONFLICT
    for (sid, d), s in schedule.items():
        if s != "OFF" and (sid, d) in required_comp_days:
            violations.append(f"COMP_CONFLICT: staff {sid} must rest on comp day {d}")

    # 5. CONSECUTIVE_WORK - check for too many consecutive work days
    for sid in active_staff_ids:
        staff_work_days = sorted([
            d for (s_id, d), shift in schedule.items()
            if s_id == sid and shift != "OFF"
        ])
        if not staff_work_days:
            continue
        consecutive = 1
        for i in range(1, len(staff_work_days)):
            gap = (staff_work_days[i] - staff_work_days[i-1]).days
            if gap == 1:  # consecutive
                consecutive += 1
                if consecutive > MAX_CONSECUTIVE_WORK_DAYS:
                    violations.append(
                        f"CONSECUTIVE_WORK: staff {sid} has {consecutive} consecutive "
                        f"work days from {staff_work_days[i-consecutive+1]}"
                    )
            else:
                consecutive = 1

    # 6. NO_REST_GAP - less than MIN_REST_DAYS_BETWEEN_SHIFTS between work days
    for sid in active_staff_ids:
        staff_work_days = sorted([
            d for (s_id, d), shift in schedule.items()
            if s_id == sid and shift != "OFF"
        ])
        for i in range(1, len(staff_work_days)):
            gap = (staff_work_days[i] - staff_work_days[i-1]).days
            if 0 < gap < MIN_REST_DAYS_BETWEEN_SHIFTS + 1:
                violations.append(
                    f"NO_REST_GAP: staff {sid} only {gap-1} rest days between "
                    f"{staff_work_days[i-1]} and {staff_work_days[i]} (need {MIN_REST_DAYS_BETWEEN_SHIFTS})"
                )

    # 7. DOUBLE_COMP - same compensation day assigned to multiple staff
    comp_by_date: Dict[date, List[int]] = {}
    for (sid, d) in required_comp_days:
        comp_by_date.setdefault(d, []).append(sid)
    for d, sids in comp_by_date.items():
        if len(sids) > 1:
            for sid in sids:
                if (sid, d) in schedule and schedule[(sid, d)] != "OFF":
                    violations.append(
                        f"DOUBLE_COMP: staff {sid} assigned on double-comp day {d}"
                    )
                    break

    # 8. L01_GAP - L01 shifts too close together
    for sid in active_staff_ids:
        l01_days = sorted([
            d for (s_id, d), shift in schedule.items()
            if s_id == sid and shift == "L01"
        ])
        for i in range(1, len(l01_days)):
            gap = (l01_days[i] - l01_days[i-1]).days
            if gap < MIN_GAP_BETWEEN_L01:
                violations.append(
                    f"L01_GAP: staff {sid} L01 on {l01_days[i-1]} and {l01_days[i]} "
                    f"only {gap} days apart (need {MIN_GAP_BETWEEN_L01})"
                )

    # 9. STAFF_UNAVAILABLE - inactive staff assigned to work
    for (sid, d), s in schedule.items():
        if s != "OFF" and sid in inactive_staff_ids:
            violations.append(f"STAFF_UNAVAILABLE: inactive staff {sid} assigned on {d}")

    # 10. EMPTY_SHIFT_SLOT - explicitly check for missing shift assignments
    for d in sorted_dates:
        for shift_type in WORK_SHIFTS:
            min_req = min_staff_per_day.get(shift_type, 0)
            if min_req > 0:
                assigned_count = sum(
                    1 for (s_id, date_key), s in schedule.items()
                    if date_key == d and s == shift_type
                )
                if assigned_count == 0:
                    violations.append(
                        f"EMPTY_SHIFT_SLOT: {d} {shift_type} has no staff assigned (required: {min_req})"
                    )

    return (len(violations) == 0, violations)

In [131]:
print("✅ Cell 4: feasibility_check loaded — 10 constraint types")

✅ Cell 4: feasibility_check loaded — 10 constraint types


============================================================
CELL 5: SCORING FUNCTION
============================================================

In [132]:
def total_score(schedule, comp_days, staff_list, dates,
                min_staff_per_day, staff_max_shifts, required_comp_days=None):
    """
    Hàm tính điểm đã tinh chỉnh: Phạt nặng thiếu người (Coverage)
    và phạt nhẹ vi phạm Fatigue/Nghỉ bù.
    """
    if required_comp_days is None:
        required_comp_days = comp_days

    workload = build_workload(schedule)

    # 1. Fairness Score
    shift_counts = list(workload.values())
    if shift_counts:
        avg_shifts = sum(shift_counts) / len(shift_counts)
        max_shifts, min_shifts = max(shift_counts), min(shift_counts)
        fairness_score = max(0.0, 100.0 * (1.0 - (max_shifts - min_shifts) / max(avg_shifts, 1)))
    else:
        fairness_score = 0.0

    # 2. Fatigue Score
    fatigue_penalties = []
    for s in staff_list:
        staff_days = sorted([d for (sid, d), shift in schedule.items() if sid == s.id and shift != "OFF"])
        penalty = 0.0
        for i in range(1, len(staff_days)):
            gap = (staff_days[i] - staff_days[i - 1]).days
            if gap <= 1: penalty += 20.0
        fatigue_penalties.append(min(penalty, 100.0))
    fatigue_score = max(0.0, 100.0 - (sum(fatigue_penalties) / max(len(fatigue_penalties), 1)))

    # 3. Coverage Score
    total_required = sum(min_staff_per_day.values()) * len(dates)
    filled = sum(1 for (_, _), s in schedule.items() if s != "OFF")
    coverage_score = min(100.0, 100.0 * filled / max(total_required, 1))

    # 4. Compensation Score
    comp_violations = sum(1 for (sid, d) in required_comp_days if schedule.get((sid, d), "OFF") != "OFF")
    comp_score = max(0.0, 100.0 * (1.0 - comp_violations / max(len(required_comp_days), 1)))

    # Tính điểm Raw theo trọng số mới
    raw_total = (WEIGHTS["fairness"] * fairness_score +
                 WEIGHTS["fatigue"] * fatigue_score +
                 WEIGHTS["coverage"] * coverage_score +
                 WEIGHTS["compensation"] * comp_score)

    # KIỂM TRA VI PHẠM RÀNG BUỘC
    is_feasible, violations = feasibility_check(
        schedule, staff_list, dates, leave_requests,
        min_staff_per_day, staff_max_shifts, required_comp_days
    )

    # CƠ CHẾ PHẠT MỚI (Dynamic Penalty):
    # - Thiếu Coverage: Trừ 10% mỗi ca thiếu
    # - Vi phạm Fatigue/Comp: Chỉ trừ 1% mỗi lỗi (ràng buộc mềm)
    coverage_gap = total_required - filled
    other_violations = len(violations) - coverage_gap

    penalty = (coverage_gap * 10.0) + (max(0, other_violations) * 1.0)
    final_total = max(0, raw_total - penalty)

    return {
        "total": final_total,
        "raw_total": raw_total,
        "fairness": fairness_score,
        "fatigue": fatigue_score,
        "coverage": coverage_score,
        "compensation": comp_score,
        "violations": len(violations)
    }

In [133]:
print("✅ Cell 5: total_score loaded — 4-dimension scoring (fairness, fatigue, coverage, compensation)")

✅ Cell 5: total_score loaded — 4-dimension scoring (fairness, fatigue, coverage, compensation)


============================================================
CELL 6: ALGORITHMS 1-3 (Greedy, Enhanced Greedy, Beam Search)
============================================================

──────────────────────────────────────────────────────────────
ALGORITHM 1: Greedy
──────────────────────────────────────────────────────────────

In [134]:
def generate_greedy_schedule(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None):
    """Greedy with expanded specialty candidate pool (top_n = len(staff))."""
    schedule, comp_days, req_comp_days = {}, set(), set()
    for d in sorted(dates):
        for shift_type in WORK_SHIFTS:
            min_req = min_staff_per_day.get(shift_type, 0)
            for _ in range(min_req):
                cands = get_specialty_compatible_candidates(schedule, staff_list, staff_max_shifts, leave_requests, d, shift_type, shift_requirements, top_n=len(staff_list))
                if cands:
                    sid = random.choice(cands[:3])
                    assign_with_overnight(schedule, comp_days, req_comp_days, sid, d, shift_type)
    return schedule, comp_days, req_comp_days

──────────────────────────────────────────────────────────────
ALGORITHM 2: Enhanced Greedy (with fatigue awareness)
──────────────────────────────────────────────────────────────

In [135]:
def generate_enhanced_greedy_schedule(staff_list, dates, leave_requests,
                                       staff_max_shifts, min_staff_per_day,
                                       shift_requirements=None):
    """Greedy + fatigue awareness: avoid consecutive days when possible."""
    schedule: Dict[Tuple[int, date], str] = {}
    comp_days: Set[Tuple[int, date]] = set()
    req_comp_days: Set[Tuple[int, date]] = set()
    leave_index = build_leave_index(leave_requests)
    last_work: Dict[int, date] = {}

    for d in sorted(dates):
        for shift_type in WORK_SHIFTS:
            min_req = min_staff_per_day.get(shift_type, 0)
            for _ in range(min_req):
                workload = build_workload(schedule)
                candidates = []

                for s in staff_list:
                    if not s.is_active:
                        continue
                    if (s.id, d) in leave_index:
                        continue
                    # FIX: check specialty match nếu có requirements
                    if shift_requirements is not None:
                        if not check_specialty_match(s, shift_requirements, d, shift_type):
                            continue
                    cnt = workload.get(s.id, 0)
                    max_allowed = staff_max_shifts.get(s.id, 5)
                    if cnt >= max_allowed:
                        continue

                    # Fatigue: prefer staff with a rest day before
                    fatigue_bonus = 0.0
                    if s.id in last_work:
                        gap = (d - last_work[s.id]).days
                        if gap >= 1:
                            fatigue_bonus = min(gap * 10, 30.0)

                    score = (100 - cnt * 10 + fatigue_bonus)
                    candidates.append((s.id, cnt, score))

                if not candidates:
                    continue

                candidates.sort(key=lambda x: x[2], reverse=True)
                top = [c[0] for c in candidates[:3]]
                sid = random.choice(top) if top else None
                if sid is None:
                    continue

                # FIX: dùng assign_with_overnight để auto block L01 → OFF
                assign_with_overnight(schedule, comp_days, req_comp_days, sid, d, shift_type)
                last_work[sid] = d

    return schedule, comp_days, req_comp_days

──────────────────────────────────────────────────────────────
ALGORITHM 3: Beam Search
──────────────────────────────────────────────────────────────

In [136]:
def solve_beam_search(staff_list, dates, leave_requests, staff_max_shifts,
                      min_staff_per_day, shift_requirements=None, beam_width=5):
    """Beam Search: keep K best states, prune rest."""
    if staff_max_shifts is None:
        staff_max_shifts = {s.id: s.max_shifts for s in staff_list}

    sorted_dates = sorted(dates)
    leave_index = build_leave_index(leave_requests)

    def score_state(sched, comp, req_comp):
        if not sched:
            return -1e9
        return total_score(sched, comp, staff_list, sorted_dates,
                           min_staff_per_day, staff_max_shifts, req_comp)["total"]

    beams = [(dict(), set(), set())]
    best_so_far = (dict(), set(), set(), -1e9)

    for work_date in sorted_dates:
        for shift_type in ["L01", "L02", "L03", "L04"]:
            next_beams = []

            for sched, comp, req_comp in beams:
                sc = score_state(sched, comp, req_comp)
                if sc > best_so_far[3]:
                    best_so_far = (dict(sched), set(comp), set(req_comp), sc)

                cands = get_specialty_compatible_candidates(
                    sched, staff_list, staff_max_shifts,
                    leave_requests, work_date, shift_type,
                    shift_requirements, top_n=5
                )

                for sid in cands:
                    ns = dict(sched)
                    nc = set(comp)
                    nrc = set(req_comp)
                    ns[(sid, work_date)] = shift_type
                    if shift_type == "L01":
                        # FIXED: Pass required arguments to calculate_compensation_date
                        cd = calculate_compensation_date(work_date, sid, leave_index)
                        nc.add((sid, cd))
                        nrc.add((sid, cd))
                    next_beams.append((ns, nc, nrc))

                if cands and random.random() < 0.1:
                    wkld = build_workload(sched)
                    for staff in staff_list:
                        if staff.id in cands or not staff.is_active:
                            continue
                        if wkld.get(staff.id, 0) >= staff_max_shifts.get(staff.id, 5):
                            continue
                        if (staff.id, work_date) in leave_index:
                            continue
                        if shift_requirements is not None:
                            if not check_specialty_match(staff, shift_requirements, work_date, shift_type):
                                continue
                        ns2 = dict(sched)
                        nc2 = set(comp)
                        nrc2 = set(req_comp)
                        ns2[(staff.id, work_date)] = shift_type
                        if shift_type == "L01":
                            # FIXED: Pass required arguments to calculate_compensation_date
                            cd = calculate_compensation_date(work_date, staff.id, leave_index)
                            nc2.add((staff.id, cd))
                            nrc2.add((staff.id, cd))
                        next_beams.append((ns2, nc2, nrc2))
                        break

            if next_beams:
                next_beams.sort(key=lambda x: score_state(*x), reverse=True)
                beams = next_beams[:beam_width]
            else:
                if best_so_far[3] >= 0:
                    beams = [(dict(best_so_far[0]), set(best_so_far[1]), set(best_so_far[2]))]
                else:
                    beams = [(dict(), set(), set())]

    if beams:
        beams.sort(key=lambda x: score_state(*x), reverse=True)
        best_s, best_c, best_rc = beams[0]
        if score_state(best_s, best_c, best_rc) < 0 and best_so_far[3] >= 0:
            best_s, best_c, best_rc = (dict(best_so_far[0]), set(best_so_far[1]), set(best_so_far[2]))
    else:
        best_s, best_c, best_rc = dict(), set(), set()

    return best_s, best_c, best_rc

In [137]:
print("✅ Cell 6: Algorithms 1-3 loaded — Greedy, Enhanced Greedy, Beam Search")

✅ Cell 6: Algorithms 1-3 loaded — Greedy, Enhanced Greedy, Beam Search


import numpy as np

# CẤU HÌNH TRỌNG SỐ MỚI: Ưu tiên tối đa cho Coverage
WEIGHTS = {
    "coverage":    0.50,  # Tăng từ 0.3 lên 0.5
    "fairness":    0.20,
    "fatigue":     0.15,
    "compensation": 0.15,
}

──────────────────────────────────────────────────────────────
ALGORITHM 4: Simulated Annealing
──────────────────────────────────────────────────────────────

In [138]:
def solve_simulated_annealing(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None, max_iter=3000):
    """SA with feasibility filter at low temperatures."""
    sched, comp, req_comp = generate_greedy_schedule(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=shift_requirements)
    sorted_dates = sorted(dates)
    leave_index = build_leave_index(leave_requests)
    current_score = total_score(sched, comp, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, req_comp)['total']
    best_state = (dict(sched), set(comp), set(req_comp))
    best_score = current_score
    T, alpha = 100.0, 0.995
    for i in range(max_iter):
        assigned = [(sid, d) for (sid, d), s in sched.items() if s != 'OFF']
        if not assigned: break
        sid, d = random.choice(assigned)
        old_s = sched[(sid, d)]
        new_s = random.choice(WORK_SHIFTS)
        if old_s == new_s: continue
        ns, nc, nrc = dict(sched), set(comp), set(req_comp)
        ns[(sid, d)] = new_s
        if old_s == 'L01':
            oc = calculate_compensation_date(d, sid, leave_index)
            nc.discard((sid, oc)); nrc.discard((sid, oc))
        if new_s == 'L01':
            nc1 = calculate_compensation_date(d, sid, leave_index)
            nc.add((sid, nc1)); nrc.add((sid, nc1))
        res = total_score(ns, nc, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, nrc)
        if T < 5.0 and res['violations'] > 0: continue
        sc = res['total']
        if sc > current_score or random.random() < math.exp((sc - current_score) / max(T, 0.01)):
            sched, comp, req_comp, current_score = ns, nc, nrc, sc
            if sc > best_score: best_score, best_state = sc, (dict(ns), set(nc), set(nrc))
        T *= alpha
    return best_state

──────────────────────────────────────────────────────────────
ALGORITHM 5: Hill Climbing
──────────────────────────────────────────────────────────────

In [139]:
def solve_hill_climbing(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None, max_iter=1000):
    """Steepest Ascent Hill Climbing with higher iterations."""
    schedule, comp, req_comp = generate_greedy_schedule(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=shift_requirements)
    sorted_dates = sorted(dates)
    leave_index = build_leave_index(leave_requests)
    best_score = total_score(schedule, comp, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, req_comp)['total']

    for _ in range(max_iter):
        assigned = [(sid, d) for (sid, d), s in schedule.items() if s != 'OFF']
        if not assigned: break
        sid, d = random.choice(assigned)
        old_s = schedule[(sid, d)]

        for new_s in WORK_SHIFTS:
            if new_s == old_s: continue
            ns, nc, nrc = dict(schedule), set(comp), set(req_comp)
            ns[(sid, d)] = new_s

            if old_s == 'L01':
                oc = calculate_compensation_date(d, sid, leave_index)
                nc.discard((sid, oc)); nrc.discard((sid, oc))
            if new_s == 'L01':
                nc1 = calculate_compensation_date(d, sid, leave_index)
                nc.add((sid, nc1)); nrc.add((sid, nc1))

            sc = total_score(ns, nc, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, nrc)['total']
            if sc > best_score:
                schedule, comp, req_comp, best_score = ns, nc, nrc, sc
                break
    return schedule, comp, req_comp

In [140]:
print("✅ Cell 7: Algorithms 4-5 loaded — Simulated Annealing, Hill Climbing")

✅ Cell 7: Algorithms 4-5 loaded — Simulated Annealing, Hill Climbing


============================================================
CELL 8: ALGORITHMS 6-7 (Random Restart HC, Genetic)
============================================================

──────────────────────────────────────────────────────────────
ALGORITHM 6: Random Restart Hill Climbing
──────────────────────────────────────────────────────────────

In [141]:
def solve_random_restart_hc(staff_list, dates, leave_requests, staff_max_shifts,
                             min_staff_per_day, shift_requirements=None,
                             num_restarts=4, max_iter_per_restart=150):
    """Random Restart HC: multiple random starts, keep best. Memory cleanup via inner fn."""
    best_schedule, best_comp, best_req = {}, set(), set()
    best_score = -1.0
    leave_index = build_leave_index(leave_requests)
    sorted_dates = sorted(dates)

    for _ in range(num_restarts):
        def _single_restart():
            sched: Dict[Tuple[int, date], str] = {}
            comp_days: Set[Tuple[int, date]] = set()
            req_comp_days: Set[Tuple[int, date]] = set()

            for d in sorted_dates:
                for shift_type in WORK_SHIFTS:
                    min_req = min_staff_per_day.get(shift_type, 0)
                    for _ in range(min_req):
                        cands = get_specialty_compatible_candidates(
                            sched, staff_list, staff_max_shifts,
                            leave_requests, d, shift_type,
                            shift_requirements, top_n=3
                        )
                        if cands:
                            sid = random.choice(cands[:3])
                            assign_with_overnight(sched, comp_days, req_comp_days, sid, d, shift_type, leave_index)
            return sched, comp_days, req_comp_days

        schedule, comp, req_comp = _single_restart()
        hc_best = total_score(schedule, comp, staff_list, sorted_dates,
                               min_staff_per_day, staff_max_shifts, req_comp)["total"]

        for _ in range(max_iter_per_restart):
            assigned = [(sid, d) for (sid, d), s in schedule.items() if s != "OFF"]
            if not assigned:
                break

            sid, d = random.choice(assigned)
            old_shift = schedule[(sid, d)]
            new_shifts = [st for st in WORK_SHIFTS if st != old_shift
                          and (sid, d) not in req_comp
                          and (sid, d) not in leave_index]
            if not new_shifts:
                continue

            new_shift = random.choice(new_shifts)
            new_sched = dict(schedule)
            new_comp = set(comp)
            new_req = set(req_comp)
            new_sched[(sid, d)] = new_shift

            if old_shift == "L01":
                # FIXED: Pass required arguments to calculate_compensation_date
                old_c = calculate_compensation_date(d, sid, leave_index)
                new_comp.discard((sid, old_c))
                new_req.discard((sid, old_c))
            if new_shift == "L01":
                # FIXED: Pass required arguments to calculate_compensation_date
                nc = calculate_compensation_date(d, sid, leave_index)
                new_comp.add((sid, nc))
                new_req.add((sid, nc))

            sc = total_score(new_sched, new_comp, staff_list, sorted_dates,
                              min_staff_per_day, staff_max_shifts, new_req)["total"]
            if sc > hc_best:
                schedule, comp, req_comp = new_sched, new_comp, new_req
                hc_best = sc

        if hc_best > best_score:
            best_schedule, best_comp, best_req = schedule, comp, req_comp
            best_score = hc_best

        del schedule, comp, req_comp

    return best_schedule, best_comp, best_req

──────────────────────────────────────────────────────────────
ALGORITHM 7: Genetic Algorithm
──────────────────────────────────────────────────────────────

In [142]:
def solve_genetic(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None, pop_size=30, generations=40):
    leave_index = build_leave_index(leave_requests)
    sorted_dates = sorted(dates)
    def _cleanup_orphans(s, c, rc):
        valid_c, valid_rc = set(), set()
        for (sid, d) in rc:
            if s.get((sid, d - timedelta(days=1))) == 'L01':
                valid_c.add((sid, d))
                valid_rc.add((sid, d))
        return s, valid_c, valid_rc
    def fitness(ind):
        s, c, rc = ind
        return total_score(s, c, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, rc)['total']
    def crossover(p1, p2):
        s1, c1, rc1 = p1
        s2, c2, rc2 = p2
        child_s = dict(s1)
        mid = len(sorted_dates) // 2
        second_half = set(sorted_dates[mid:])
        for (sid, d), st in s2.items():
            if d in second_half:
                child_s[(sid, d)] = st
        ns, nc, nrc = _cleanup_orphans(child_s, set(), set(rc1) | set(rc2))
        return ns, nc, nrc
    def mutate(ind):
        s, c, rc = ind
        ns = dict(s)
        assigned = [(sid, d) for (sid, d), st in ns.items() if st != 'OFF']
        if assigned:
            sid, d = random.choice(assigned)
            ns[(sid, d)] = random.choice(WORK_SHIFTS)
        return _cleanup_orphans(ns, set(), rc)
    population = [generate_greedy_schedule(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements) for _ in range(pop_size)]
    best_ind = max(population, key=fitness)
    for gen in range(generations):
        scored = sorted(population, key=fitness, reverse=True)
        new_pop = scored[:pop_size // 2]
        while len(new_pop) < pop_size:
            p1, p2 = random.sample(scored[:10], 2)
            child = crossover(p1, p2)
            if random.random() < 0.3: child = mutate(child)
            new_pop.append(child)
        population = new_pop
        current_best = max(population, key=fitness)
        if fitness(current_best) > fitness(best_ind): best_ind = current_best
    return best_ind

In [143]:
print("✅ Cell 8: Algorithms 6-7 loaded — Random Restart HC, Genetic")

✅ Cell 8: Algorithms 6-7 loaded — Random Restart HC, Genetic


============================================================
CELL 9: ALGORITHMS 8-9 (Tabu Search, Hybrid)
============================================================

──────────────────────────────────────────────────────────────
ALGORITHM 8: Tabu Search
──────────────────────────────────────────────────────────────

In [144]:
def solve_tabu_search(staff_list, dates, leave_requests, staff_max_shifts,
                     min_staff_per_day, shift_requirements=None, max_iter=300, tabu_size=15):
    """Tabu Search: avoid revisiting recent neighbors."""
    schedule, comp, req_comp = generate_greedy_schedule(
        staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day,
        shift_requirements=shift_requirements
    )
    leave_index = build_leave_index(leave_requests)
    sorted_dates = sorted(dates)

    best_schedule = dict(schedule)
    best_comp = set(comp)
    best_req = set(req_comp)
    best_score = total_score(schedule, comp, staff_list, sorted_dates,
                              min_staff_per_day, staff_max_shifts, req_comp)["total"]

    tabu_list: List[Tuple[Tuple[int, date], str]] = []
    current_score = best_score

    for _ in range(max_iter):
        assigned = [(sid, d) for (sid, d), s in schedule.items() if s != "OFF"]
        if not assigned:
            break

        best_neighbor = None
        best_neighbor_score = -1.0
        best_move = None

        sample = random.sample(assigned, min(10, len(assigned)))

        for sid, d in sample:
            old_shift = schedule[(sid, d)]
            new_shifts = [st for st in WORK_SHIFTS if st != old_shift
                          and (sid, d) not in req_comp
                          and (sid, d) not in leave_index]

            for new_shift in new_shifts:
                ns = dict(schedule)
                nc = set(comp)
                nrc = set(req_comp)
                ns[(sid, d)] = new_shift

                if old_shift == "L01":
                    # FIXED: Pass sid and leave_index
                    old_c = calculate_compensation_date(d, sid, leave_index)
                    nc.discard((sid, old_c))
                    nrc.discard((sid, old_c))
                if new_shift == "L01":
                    # FIXED: Pass sid and leave_index
                    nc2 = calculate_compensation_date(d, sid, leave_index)
                    nc.add((sid, nc2))
                    nrc.add((sid, nc2))

                move = ((sid, d), old_shift)
                is_tabu = move in tabu_list and total_score(ns, nc, staff_list, sorted_dates,
                                                            min_staff_per_day, staff_max_shifts, nrc)["total"] <= best_score

                if not is_tabu:
                    sc = total_score(ns, nc, staff_list, sorted_dates,
                                      min_staff_per_day, staff_max_shifts, nrc)["total"]
                    if sc > best_neighbor_score:
                        best_neighbor = (ns, nc, nrc)
                        best_neighbor_score = sc
                        best_move = (sid, d, old_shift, new_shift)

        if best_neighbor:
            schedule, comp, req_comp = best_neighbor
            current_score = best_neighbor_score

            if current_score > best_score:
                best_schedule, best_comp, best_req = dict(schedule), set(comp), set(req_comp)
                best_score = current_score

            if best_move is not None:
                sid, d, old_shift, new_shift = best_move
                tabu_list.append(((sid, d), new_shift))
                if len(tabu_list) > tabu_size:
                    tabu_list.pop(0)

    return best_schedule, best_comp, best_req

──────────────────────────────────────────────────────────────
ALGORITHM 9: Hybrid (Greedy + Local Search)
──────────────────────────────────────────────────────────────

In [145]:
def solve_hybrid(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None, max_iter=2000):
    # Hybrid: Greedy base + SA refinement instead of Beam Search duplication
    schedule, comp, req_comp = generate_enhanced_greedy_schedule(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements)
    sorted_dates = sorted(dates)
    leave_index = build_leave_index(leave_requests)
    best_state = (dict(schedule), set(comp), set(req_comp))
    best_score = total_score(schedule, comp, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, req_comp)['total']
    T = 50.0
    for _ in range(max_iter):
        assigned = [(sid, d) for (sid, d), s in schedule.items() if s != 'OFF']
        if not assigned: break
        sid, d = random.choice(assigned)
        new_shift = random.choice(WORK_SHIFTS)
        ns, nc, nrc = dict(schedule), set(comp), set(req_comp)
        ns[(sid, d)] = new_shift
        if new_shift == 'L01':
            # FIXED: Pass required staff_id and leave_index
            cd = calculate_compensation_date(d, sid, leave_index)
            nc.add((sid, cd)); nrc.add((sid, cd))
        sc = total_score(ns, nc, staff_list, sorted_dates, min_staff_per_day, staff_max_shifts, nrc)['total']
        if sc > best_score or random.random() < math.exp((sc - best_score) / max(T, 0.01)):
            schedule, comp, req_comp = ns, nc, nrc
            if sc > best_score:
                best_score = sc
                best_state = (dict(ns), set(nc), set(nrc))
        T *= 0.999
    return best_state

In [146]:
def solve_backtracking(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None):
    """Backtracking Solver: Chinh quy, tim loi giai chinh xac cho bai toan CSP."""
    sorted_dates = sorted(dates)
    leave_index = build_leave_index(leave_requests)
    schedule = {}
    comp_days = set()
    req_comp_days = set()

    # Buoc 1 & 2: Xep L01 truoc vi no sinh ra rang buoc nghi bu cho ca thang
    def assign_l01_recursive(date_idx):
        if date_idx == len(sorted_dates):
            return True

        current_date = sorted_dates[date_idx]
        # Tim ung vien cho L01
        candidates = get_specialty_compatible_candidates(schedule, staff_list, staff_max_shifts, leave_requests, current_date, 'L01', shift_requirements, top_n=len(staff_list))
        random.shuffle(candidates) # Them tinh ngau nhien cho moi lan chay

        for sid in candidates:
            if can_assign_shift(schedule, leave_index, req_comp_days, sid, current_date, 'L01'):
                # Thu assign
                old_req_comp = set(req_comp_days)
                assign_with_overnight(schedule, comp_days, req_comp_days, sid, current_date, 'L01', leave_index)

                if assign_l01_recursive(date_idx + 1):
                    return True

                # Backtrack
                del schedule[(sid, current_date)]
                req_comp_days = old_req_comp
        return False

    # Thuc thi giai doan 1 (L01)
    if not assign_l01_recursive(0):
        # Neu L01 khong the xep hoan hao, van tiep tuc de Greedy ho tro cac ca khac
        pass

    # Buoc 3: Dien cac ca con lai (L02, L03, L04) bang Greedy/Heuristic tren nen L01 da co
    for d in sorted_dates:
        for shift_type in ["L02", "L03", "L04"]:
            min_req = min_staff_per_day.get(shift_type, 0)
            for _ in range(min_req):
                cands = get_specialty_compatible_candidates(schedule, staff_list, staff_max_shifts, leave_requests, d, shift_type, shift_requirements)
                if cands:
                    sid = cands[0] # Chon nguoi it cong nhat
                    if can_assign_shift(schedule, leave_index, req_comp_days, sid, d, shift_type):
                        schedule[(sid, d)] = shift_type

    return schedule, comp_days, req_comp_days

In [147]:
from ortools.sat.python import cp_model

def solve_ortools_cp_sat(staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day, shift_requirements=None):
    """Google OR-Tools CP-SAT Solver: Tối ưu hóa ràng buộc chính quy."""
    model = cp_model.CpModel()
    num_staff = len(staff_list)
    num_days = len(dates)
    staff_ids = [s.id for s in staff_list]
    leave_index = build_leave_index(leave_requests)

    # Variables: x[s, d, shift] = 1 nếu nhân viên s làm ca shift ngày d
    x = {}
    for sid in staff_ids:
        for d_idx, d in enumerate(dates):
            for shift in WORK_SHIFTS:
                x[sid, d_idx, shift] = model.NewBoolVar(f'x_{sid}_{d_idx}_{shift}')

    # Constraint 1: Mỗi nhân viên làm tối đa 1 ca mỗi ngày
    for sid in staff_ids:
        for d_idx in range(num_days):
            model.Add(sum(x[sid, d_idx, s] for s in WORK_SHIFTS) <= 1)

    # Constraint 2: Định mức quân số (Coverage)
    for d_idx in range(num_days):
        for shift in WORK_SHIFTS:
            min_req = min_staff_per_day.get(shift, 0)
            model.Add(sum(x[sid, d_idx, shift] for sid in staff_ids) >= min_req)

    # Constraint 3: Nghỉ bù sau L01 (Simplified for CP-SAT core)
    for sid in staff_ids:
        for d_idx in range(num_days - 1):
            model.Add(x[sid, d_idx, 'L01'] + x[sid, d_idx + 1, 'L01'] <= 1)

    # Objective: Tối ưu hóa sự công bằng (phân bổ đều ca)
    total_shifts_vars = []
    for sid in staff_ids:
        num_shifts = model.NewIntVar(0, num_days, f'shifts_{sid}')
        model.Add(num_shifts == sum(x[sid, d_idx, s] for d_idx in range(num_days) for s in WORK_SHIFTS))
        total_shifts_vars.append(num_shifts)

    # Tìm cách để số ca của mọi người gần nhau nhất
    min_shifts = model.NewIntVar(0, num_days, 'min_shifts')
    max_shifts = model.NewIntVar(0, num_days, 'max_shifts')
    model.AddMinEquality(min_shifts, total_shifts_vars)
    model.AddMaxEquality(max_shifts, total_shifts_vars)
    model.Minimize(max_shifts - min_shifts)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10.0
    status = solver.Solve(model)

    schedule, comp_days, req_comp_days = {}, set(), set()
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        for sid in staff_ids:
            for d_idx, d in enumerate(dates):
                for shift in WORK_SHIFTS:
                    if solver.Value(x[sid, d_idx, shift]):
                        assign_with_overnight(schedule, comp_days, req_comp_days, sid, d, shift, leave_index)

    return schedule, comp_days, req_comp_days

In [148]:
# ============================================================
# CELL 10: QUY TRÌNH 7 BƯỚC NGHIỆP VỤ (CHÍNH THỨC)
# ============================================================

def hospital_scheduling_pipeline(staff_list, dates, leave_requests, min_staff_per_day, staff_max_shifts):
    """Thực thi quy trình xếp lịch 7 bước theo đặc tả v5."""
    print("--- BẮT ĐẦU QUY TRÌNH XẾP LỊCH 7 BƯỚC ---")

    # B1: Đọc dữ liệu nền tảng (Đã có staff_list, leave_requests)
    leave_index = build_leave_index(leave_requests)

    # B2 & B3: Sử dụng Backtracking Solver để xếp L01 trước, sau đó là L02-L04
    # Đây là lõi thuật toán M07
    schedule, comp_days, req_comp_days = solve_backtracking(
        staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day
    )

    # B4: Quét toàn bộ ràng buộc (Feasibility Check)
    is_feasible, violations = feasibility_check(
        schedule, staff_list, dates, leave_requests,
        min_staff_per_day, staff_max_shifts, req_comp_days
    )

    # B5: Tính toán điểm tối ưu (Scoring)
    scores = total_score(schedule, comp_days, staff_list, dates,
                         min_staff_per_day, staff_max_shifts, req_comp_days)

    # B6 & B7: Xuất bản nháp (Draft) và Log kết quả
    print(f"Kết quả: {'Hoàn hảo' if is_feasible else 'Có vi phạm (Chặn mềm)'}")
    print(f"Tổng điểm tối ưu: {scores['total']:.2f}")
    if violations:
        print(f"Danh sách vi phạm lưu DB (conflict_note): {violations[:3]}...")

    return schedule, scores

In [149]:
print("✅ Cell 9: Algorithms 8-9 loaded — Tabu Search, Hybrid")
print("✅ All 9 algorithms loaded: Greedy, Enhanced Greedy, Beam Search, SA, HC, RRHC, Genetic, Tabu, Hybrid")

✅ Cell 9: Algorithms 8-9 loaded — Tabu Search, Hybrid
✅ All 9 algorithms loaded: Greedy, Enhanced Greedy, Beam Search, SA, HC, RRHC, Genetic, Tabu, Hybrid


============================================================
CELL 10: BENCHMARK RUNNER
============================================================

In [150]:
# ─────────────────────────────────────────────────────────
# VALIDATION: Ensure all required data is available BEFORE running
# ─────────────────────────────────────────────────────────
required_vars = {
    'staff_list': 'Cell 2',
    'dates': 'Cell 2',
    'leave_requests': 'Cell 2',
    'staff_max_shifts': 'Cell 2',
    'min_staff_per_day': 'Cell 2',
    'num_days': 'Cell 2',
}

In [151]:
missing = []
for var_name, cell_hint in required_vars.items():
    try:
        _ = eval(var_name)
    except NameError:
        missing.append(f"  • {var_name} ← {cell_hint}")

In [152]:
if missing:
    error_msg = (
        "\n" + "=" * 60 + "\n"
        "❌ BENCHMARK SKIPPED: Missing required data!\n"
        "=" * 60 + "\n"
        "The following variables are not defined:\n"
        + "\n".join(missing) + "\n\n"
        "▶ REQUIRED CELL ORDER:\n"
        "   Cell 1  → Load data models\n"
        "   Cell 2  → Create test data\n"
        "   Cell 3  → Utility functions\n"
        "   Cell 4  → Feasibility check\n"
        "   Cell 5  → Scoring function\n"
        "   Cell 6-9 → Load algorithms\n"
        "   Cell 10 → Run benchmark\n\n"
        "   Please run: Cell → Run All Above\n"
        + "=" * 60 + "\n"
    )
    print(error_msg)
    raise RuntimeError("Missing required variables: " + ", ".join(list(required_vars.keys())))

In [153]:
# Validate algorithms
required_algos = [
    'generate_greedy_schedule',
    'generate_enhanced_greedy_schedule',
    'solve_beam_search',
    'solve_simulated_annealing',
    'solve_hill_climbing',
    'solve_random_restart_hc',
    'solve_genetic',
    'solve_tabu_search',
    'solve_hybrid',
]
missing_algos = [fn for fn in required_algos if fn not in dir()]
if missing_algos:
    print(f"\n⚠ WARNING: Missing algorithms: {missing_algos}")
    print("   Please ensure Cells 6-9 have been run.\n")

In [154]:
print("✅ Validation passed: all data and algorithms available")
print(f"   Data: {len(staff_list)} staff, {len(dates)} days, {len(leave_requests)} leave requests\n")

✅ Validation passed: all data and algorithms available
   Data: 10 staff, 30 days, 30 leave requests



─────────────────────────────────────────────────────────
BENCHMARK EXECUTION
─────────────────────────────────────────────────────────

In [155]:
all_schedules = {}
times = {}
results = []

In [156]:
def run_algorithm(name, fn):
    start = time.perf_counter()
    result = fn()
    elapsed = (time.perf_counter() - start) * 1000.0
    return result, elapsed

In [157]:
def _is_empty(sched):
    return not sched or sum(1 for (_, _), v in sched.items() if v != "OFF") == 0

In [158]:
def _store(name, sched, comp, req_comp, ms):
    if _is_empty(sched):
        print(f"  [EMPTY] {name} — using greedy fallback")
        sched, comp, req_comp = generate_greedy_schedule(
            staff_list, dates, leave_requests, staff_max_shifts, min_staff_per_day
        )
    all_schedules[name] = (sched, comp, req_comp)
    times[name] = ms

In [159]:
def _run(name, fn):
    (sched, comp, req_comp), ms = run_algorithm(name, fn)
    _store(name, sched, comp, req_comp, ms)

In [160]:
# --- 1. Greedy ---
def wrapper_greedy():
    return generate_greedy_schedule(staff_list, dates, leave_requests,
                                     staff_max_shifts, min_staff_per_day,
                                     shift_requirements=shift_requirements)
_run("Greedy", wrapper_greedy)

In [161]:
# --- 2. Enhanced Greedy ---
def wrapper_enhanced_greedy():
    return generate_enhanced_greedy_schedule(staff_list, dates, leave_requests,
                                            staff_max_shifts, min_staff_per_day,
                                            shift_requirements=shift_requirements)
_run("Enhanced Greedy", wrapper_enhanced_greedy)

In [162]:
# --- 3. Beam Search ---
def wrapper_beam_search():
    return solve_beam_search(staff_list, dates, leave_requests, staff_max_shifts,
                              min_staff_per_day, shift_requirements=shift_requirements,
                              beam_width=5)
_run("Beam Search", wrapper_beam_search)

In [163]:
# --- 4. Simulated Annealing ---
def wrapper_sa():
    return solve_simulated_annealing(staff_list, dates, leave_requests, staff_max_shifts,
                                      min_staff_per_day, shift_requirements=shift_requirements,
                                      max_iter=3000)
_run("Simulated Annealing", wrapper_sa)

In [164]:
# --- 5. Hill Climbing ---
def wrapper_hc():
    return solve_hill_climbing(staff_list, dates, leave_requests, staff_max_shifts,
                                min_staff_per_day, shift_requirements=shift_requirements,
                                max_iter=300)
_run("Hill Climbing", wrapper_hc)

In [165]:
# --- 6. Random Restart HC ---
def wrapper_rrhc():
    return solve_random_restart_hc(staff_list, dates, leave_requests, staff_max_shifts,
                                    min_staff_per_day, shift_requirements=shift_requirements,
                                    num_restarts=4, max_iter_per_restart=150)
_run("Random Restart HC", wrapper_rrhc)

In [166]:
# --- 7. Genetic ---
def wrapper_genetic():
    return solve_genetic(staff_list, dates, leave_requests, staff_max_shifts,
                          min_staff_per_day, shift_requirements=shift_requirements,
                          pop_size=30, generations=40)
_run("Genetic", wrapper_genetic)

In [167]:
# --- 8. Tabu Search ---
def wrapper_tabu():
    return solve_tabu_search(staff_list, dates, leave_requests, staff_max_shifts,
                              min_staff_per_day, shift_requirements=shift_requirements,
                              max_iter=300, tabu_size=15)
_run("Tabu Search", wrapper_tabu)

In [168]:
# --- 9. Hybrid ---
def wrapper_hybrid():
    return solve_hybrid(staff_list, dates, leave_requests, staff_max_shifts,
                        min_staff_per_day, shift_requirements=shift_requirements,
                        max_iter=2000)
_run("Hybrid", wrapper_hybrid)

In [169]:
print(f"\n✅ Benchmark complete — {len(all_schedules)} algorithms evaluated.")


✅ Benchmark complete — 9 algorithms evaluated.


In [170]:
# Reset results and re-run scoring with new logic
results = []
for name, (sched, comp, req_comp) in all_schedules.items():
    scores = total_score(sched, comp, staff_list, dates,
                         min_staff_per_day, staff_max_shifts, req_comp)

    assigned = sum(1 for (_, _), s in sched.items() if s != "OFF")
    t = times.get(name, 0)

    results.append({
        "name": name,
        "time_ms": round(t, 1),
        "total": scores["total"],
        "fairness": scores["fairness"],
        "fatigue": scores["fatigue"],
        "coverage": scores["coverage"],
        "compensation": scores["compensation"],
        "assigned": assigned
    })

# Display new ranking
import pandas as pd
df_new = pd.DataFrame(results)
df_new = df_new.sort_values('total', ascending=False).reset_index(drop=True)
df_new['Rank'] = range(1, len(df_new) + 1)
print("=== NEW RANKING (Prioritizing Coverage) ===")
print(df_new[['Rank', 'name', 'total', 'coverage', 'assigned']].to_string(index=False))

=== NEW RANKING (Prioritizing Coverage) ===
 Rank                name      total   coverage  assigned
    1             Genetic 502.825397 100.000000       189
    2              Hybrid  62.231034 100.000000       120
    3     Enhanced Greedy  61.696296 100.000000       120
    4         Beam Search  49.857563  99.166667       119
    5       Hill Climbing  49.343277  99.166667       119
    6   Random Restart HC  35.453923  98.333333       118
    7 Simulated Annealing  21.971612  97.500000       117
    8              Greedy   5.475862  96.666667       116
    9         Tabu Search   0.000000  93.333333       112


In [197]:
hybrid_sched, hybrid_comp, hybrid_req = all_schedules['Hybrid']
is_feasible, violations = feasibility_check(
    hybrid_sched, staff_list, dates, leave_requests,
    min_staff_per_day, staff_max_shifts, hybrid_req
)

print(f"=== DETAILED VIOLATIONS FOR HYBRID ALGORITHM ({len(violations)} total) ===")
for i, v in enumerate(violations, 1):
    print(f"{i}. {v}")

=== DETAILED VIOLATIONS FOR HYBRID ALGORITHM (23 total) ===
1. COMP_CONFLICT: staff 2 must rest on comp day 2024-06-12
2. COMP_CONFLICT: staff 7 must rest on comp day 2024-06-19
3. COMP_CONFLICT: staff 2 must rest on comp day 2024-06-25
4. NO_REST_GAP: staff 2 only 0 rest days between 2024-06-24 and 2024-06-25 (need 1)
5. NO_REST_GAP: staff 3 only 0 rest days between 2024-06-01 and 2024-06-02 (need 1)
6. NO_REST_GAP: staff 4 only 0 rest days between 2024-06-01 and 2024-06-02 (need 1)
7. NO_REST_GAP: staff 4 only 0 rest days between 2024-06-22 and 2024-06-23 (need 1)
8. NO_REST_GAP: staff 5 only 0 rest days between 2024-06-02 and 2024-06-03 (need 1)
9. NO_REST_GAP: staff 6 only 0 rest days between 2024-06-03 and 2024-06-04 (need 1)
10. NO_REST_GAP: staff 8 only 0 rest days between 2024-06-05 and 2024-06-06 (need 1)
11. NO_REST_GAP: staff 9 only 0 rest days between 2024-06-09 and 2024-06-10 (need 1)
12. NO_REST_GAP: staff 9 only 0 rest days between 2024-06-10 and 2024-06-11 (need 1)
13. 

============================================================
CELL 11: SCORE CALCULATION
============================================================

In [171]:
if 'results' not in dir() or not results:
    results = []

In [172]:
print(f"\n{'Algorithm':<22} {'Time(ms)':<10} {'Total':<8} {'Fair':<8} {'Fatigue':<9} {'Coverage':<10} {'Comp.':<8} {'Assigned':<10}")
print("-" * 95)


Algorithm              Time(ms)   Total    Fair     Fatigue   Coverage   Comp.    Assigned  
-----------------------------------------------------------------------------------------------


In [173]:
for name, (sched, comp, req_comp) in all_schedules.items():
    scores = total_score(sched, comp, staff_list, dates,
                         min_staff_per_day, staff_max_shifts, req_comp)

    assigned = sum(1 for (_, _), s in sched.items() if s != "OFF")
    t = times.get(name, 0)

    results.append({
        "name": name,
        "time_ms": round(t, 1),
        "total": scores["total"],
        "fairness": scores["fairness"],
        "fatigue": scores["fatigue"],
        "coverage": scores["coverage"],
        "compensation": scores["compensation"],
        "assigned": assigned
    })

    print(f"{name:<22} {t:<10.1f} {scores['total']:<8.2f} {scores['fairness']:<8.2f} "
          f"{scores['fatigue']:<9.2f} {scores['coverage']:<10.2f} {scores['compensation']:<8.2f} {assigned:<10}")

Greedy                 2.0        5.48     91.38    56.00     96.67      79.31    116       
Enhanced Greedy        2.0        61.70    83.33    72.00     100.00     81.48    120       
Beam Search            9999.0     49.86    91.60    86.00     99.17      57.14    119       
Simulated Annealing    9929.8     21.97    91.45    70.00     97.50      71.43    117       
Hill Climbing          2711.7     49.34    91.60    72.00     99.17      78.57    119       
Random Restart HC      1008.3     35.45    91.53    66.00     98.33      81.48    118       
Genetic                8795.7     502.83   89.42    0.00      100.00     50.00    189       
Tabu Search            18399.6    0.00     91.07    66.00     93.33      79.31    112       
Hybrid                 5033.1     62.23    75.00    74.00     100.00     89.66    120       


In [174]:
print("-" * 95)

-----------------------------------------------------------------------------------------------


In [175]:
# Coverage check
print("\n=== COVERAGE CHECK (should be <= 100) ===")
all_ok = True
for r in results:
    status = "OK" if r["coverage"] <= 100 else "BUG!"
    if r["coverage"] > 100:
        all_ok = False
    print(f"  {r['name']:<22}: coverage = {r['coverage']:.2f}  [{status}]")


=== COVERAGE CHECK (should be <= 100) ===
  Greedy                : coverage = 96.67  [OK]
  Enhanced Greedy       : coverage = 100.00  [OK]
  Beam Search           : coverage = 99.17  [OK]
  Simulated Annealing   : coverage = 97.50  [OK]
  Hill Climbing         : coverage = 99.17  [OK]
  Random Restart HC     : coverage = 98.33  [OK]
  Genetic               : coverage = 100.00  [OK]
  Tabu Search           : coverage = 93.33  [OK]
  Hybrid                : coverage = 100.00  [OK]
  Greedy                : coverage = 96.67  [OK]
  Enhanced Greedy       : coverage = 100.00  [OK]
  Beam Search           : coverage = 99.17  [OK]
  Simulated Annealing   : coverage = 97.50  [OK]
  Hill Climbing         : coverage = 99.17  [OK]
  Random Restart HC     : coverage = 98.33  [OK]
  Genetic               : coverage = 100.00  [OK]
  Tabu Search           : coverage = 93.33  [OK]
  Hybrid                : coverage = 100.00  [OK]


In [176]:
print()
if all_ok:
    print("=> PASS: Tat ca coverage deu nam trong khoang [0, 100]!")
else:
    print("=> FAIL: Co coverage vuot 100!")


=> PASS: Tat ca coverage deu nam trong khoang [0, 100]!


In [177]:
best = max(results, key=lambda x: x["total"])
print(f"\n>>> WINNER: {best['name']} voi total score = {best['total']:.2f}")


>>> WINNER: Genetic voi total score = 502.83


============================================================
CELL 12: COMPARISON TABLE (Pandas)
============================================================

In [178]:
import pandas as pd
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 160)

In [179]:
df = pd.DataFrame(results)
df = df.sort_values('total', ascending=False).reset_index(drop=True)
df['Rank'] = range(1, len(df) + 1)
df.columns = ['Algorithm', 'Time(ms)', 'TOTAL', 'Fairness', 'Fatigue', 'Coverage', 'Compensation', 'Assigned', 'Rank']
print(df.to_string(index=False))

          Algorithm  Time(ms)      TOTAL  Fairness  Fatigue   Coverage  Compensation  Assigned  Rank
            Genetic    8795.7 502.825397 89.417989      0.0 100.000000     50.000000       189     1
            Genetic    8795.7 502.825397 89.417989      0.0 100.000000     50.000000       189     2
             Hybrid    5033.1  62.231034 75.000000     74.0 100.000000     89.655172       120     3
             Hybrid    5033.1  62.231034 75.000000     74.0 100.000000     89.655172       120     4
    Enhanced Greedy       2.0  61.696296 83.333333     72.0 100.000000     81.481481       120     5
    Enhanced Greedy       2.0  61.696296 83.333333     72.0 100.000000     81.481481       120     6
        Beam Search    9999.0  49.857563 91.596639     86.0  99.166667     57.142857       119     7
        Beam Search    9999.0  49.857563 91.596639     86.0  99.166667     57.142857       119     8
      Hill Climbing    2711.7  49.343277 91.596639     72.0  99.166667     78.571429       

In [180]:
print("\n=== DIEM TRUNG BINH THEO METRIC ===")
for col in ['Fairness', 'Fatigue', 'Coverage', 'Compensation']:
    avg = df[col].mean()
    print(f"  {col}: {avg:.2f}")


=== DIEM TRUNG BINH THEO METRIC ===
  Fairness: 88.49
  Fatigue: 62.44
  Coverage: 98.24
  Compensation: 74.26


============================================================
CELL 13: VISUALIZATION (Matplotlib)
============================================================

In [181]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    sorted_results = sorted(results, key=lambda x: x['total'])
    algo_names = [r['name'] for r in sorted_results]
    totals = [r['total'] for r in sorted_results]

    colors = plt.cm.tab10(np.linspace(0, 1, len(algo_names)))

    # Chart 1: Total score bar chart
    bars = axes[0].barh(algo_names, totals, color=colors)
    axes[0].set_xlabel('Total Score (0-100)')
    axes[0].set_title('So sanh tong diem (sorted ascending)')
    axes[0].set_xlim(0, 105)
    for bar, val in zip(bars, totals):
        axes[0].text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                    f'{val:.1f}', va='center', fontsize=9)

    # Chart 2: Grouped bar
    metrics = ['fairness', 'fatigue', 'coverage', 'compensation']
    metric_labels = ['Fairness', 'Fatigue', 'Coverage', 'Compensation']
    x = np.arange(len(algo_names))
    width = 0.18

    for i, (m, label) in enumerate(zip(metrics, metric_labels)):
        values = [r[m] for r in sorted_results]
        axes[1].bar(x + i * width, values, width, label=label, alpha=0.85)

    axes[1].set_xticks(x + width * 1.5)
    axes[1].set_xticklabels(algo_names, rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('Score (0-100)')
    axes[1].set_title('Chi tiet diem thanh phan')
    axes[1].legend(fontsize=8)
    axes[1].set_ylim(0, 110)
    axes[1].axhline(y=100, color='red', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig('benchmark_chart.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Chart saved: benchmark_chart.png")
except Exception as e:
    print(f"Loi ve chart: {e}")

/tmp/ipykernel_3063/3415274402.py:41: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


Chart saved: benchmark_chart.png


In [182]:
import math

In [183]:
print("=== WORKLOAD DISTRIBUTION PER STAFF ===\n")
print(f"{'Algorithm':<22} " + "".join([f"{s.name:<8}" for s in staff_list]) + " {'StdDev':<8} {'Min':<5} {'Max':<5}")
print("-" * 75)

=== WORKLOAD DISTRIBUTION PER STAFF ===

Algorithm              BS.A    BS.B    BS.C    BS.D    BS.E    BS.F    BS.G    BS.H    BS.I    BS.J     {'StdDev':<8} {'Min':<5} {'Max':<5}
---------------------------------------------------------------------------


In [184]:
for name, (sched, comp, req_comp) in all_schedules.items():
    workloads = {}
    for staff in staff_list:
        workloads[staff.id] = sum(
            1 for (sid, _), s in sched.items()
            if sid == staff.id and s != "OFF"
        )

    vals = list(workloads.values())
    mean_val = sum(vals) / len(vals) if vals else 0
    std_val = math.sqrt(sum((v - mean_val) ** 2 for v in vals) / len(vals)) if vals else 0
    min_val = min(vals) if vals else 0
    max_val = max(vals) if vals else 0

    row = f"{name:<22} "
    row += "".join([f"{workloads.get(s.id, 0):<8}" for s in staff_list])
    row += f" {std_val:<8.2f} {min_val:<5} {max_val:<5}"
    print(row)

Greedy                 12      12      12      11      12      12      12      11      11      11       0.49     11    12   
Enhanced Greedy        13      12      12      12      12      12      11      12      12      12       0.45     11    13   
Beam Search            12      12      12      12      12      12      12      12      12      11       0.30     11    12   
Simulated Annealing    11      12      12      12      11      12      12      12      12      11       0.46     11    12   
Hill Climbing          12      12      12      12      12      12      12      12      12      11       0.30     11    12   
Random Restart HC      12      12      12      12      12      12      12      11      12      11       0.40     11    12   
Genetic                18      18      18      18      19      20      20      19      20      19       0.83     18    20   
Tabu Search            12      12      11      11      11      11      11      11      11      11       0.40     11    12   


In [185]:
print("-" * 75)

---------------------------------------------------------------------------


============================================================
CELL 15: SAVE RESULTS TO JSON
============================================================

In [186]:
benchmark_output = {
    "metadata": {
        "num_staff": len(staff_list),
        "num_days": num_days,
        "required_count": required_count,
        "seed": 42,
        "timestamp": str(date.today())
    },
    "results": results,
    "scores_detail": {}
}

In [187]:
for name, (sched, comp, req_comp) in all_schedules.items():
    scores = total_score(sched, comp, staff_list, sorted(dates), min_staff_per_day, staff_max_shifts, req_comp)
    benchmark_output["scores_detail"][name] = {
        **scores,
        "time_ms": times.get(name, 0),
        "assigned": sum(1 for (_, _), s in sched.items() if s != "OFF")
    }

In [188]:
with open("benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_output, f, indent=2, ensure_ascii=False)

In [189]:
print("✅ Da luu ket qua vao benchmark_results.json")
print(f"   Tong so thuat toan: {len(results)}")
print(f"   Tong so assignment: {required_count}")

✅ Da luu ket qua vao benchmark_results.json
   Tong so thuat toan: 18
   Tong so assignment: 120


============================================================
CELL 16: CONCLUSION & RANKING
============================================================

### **STEP B4 & B5: CONFLICT RE-CHECK & API STANDARDIZATION**
This section implements the logic to clear conflict flags after manual edits and formats the final payload for the Frontend team.

In [198]:
def update_schedule_conflicts(schedule, staff_list, dates, leave_requests, min_staff_per_day, staff_max_shifts, req_comp_days):
    """
    Re-evaluates the schedule and returns a structured list of assignments with conflict flags.
    Matches Step B4 and Step B5 requirements.
    """
    # 1. Run the full feasibility check
    is_feasible, violations = feasibility_check(
        schedule, staff_list, dates, leave_requests,
        min_staff_per_day, staff_max_shifts, req_comp_days
    )

    # 2. Map violations to specific assignments for the UI
    # In a real DB, this would update rows. Here we return an enriched list.
    enriched_assignments = []

    for d in sorted(dates):
        for s in staff_list:
            shift = schedule.get((s.id, d), "OFF")
            if shift == "OFF": continue

            # Find if this specific (staff, date) has a violation related to it
            # Simple string matching for demo purposes
            relevant_violations = [v for v in violations if f"staff {s.id}" in v and str(d) in v]

            enriched_assignments.append({
                "date": str(d),
                "staff_id": s.id,
                "staff_name": s.name,
                "shift_type": shift,
                "has_conflict": len(relevant_violations) > 0,
                "conflict_note": "; ".join(relevant_violations) if relevant_violations else None
            })

    return enriched_assignments, is_feasible, violations

In [199]:
def format_schedule_to_json(enriched_assignments, scores, total_violations):
    """
    Wraps the Hybrid algorithm's output into a clean JSON for Frontend consumption.
    """
    payload = {
        "status": "DRAFT",
        "metadata": {
            "total_score": round(scores['total'], 2),
            "coverage_pct": scores['coverage'],
            "total_conflicts": len(total_violations),
            "generated_at": str(date.today())
        },
        "assignments": enriched_assignments
    }
    return json.dumps(payload, indent=2, ensure_ascii=False)

# --- Execution Demo ---
hybrid_sched, hybrid_comp, hybrid_req = all_schedules['Hybrid']
hybrid_scores = total_score(hybrid_sched, hybrid_comp, staff_list, dates, min_staff_per_day, staff_max_shifts, hybrid_req)

enriched_data, feasible, all_v = update_schedule_conflicts(
    hybrid_sched, staff_list, dates, leave_requests, min_staff_per_day, staff_max_shifts, hybrid_req
)

json_payload = format_schedule_to_json(enriched_data, hybrid_scores, all_v)
print("✅ API Payload Ready for Frontend!")
print(f"Sample (First 2 assignments):\n{json.dumps(enriched_data[:2], indent=2)}")

✅ API Payload Ready for Frontend!
Sample (First 2 assignments):
[
  {
    "date": "2024-06-01",
    "staff_id": 1,
    "staff_name": "BS.A",
    "shift_type": "L02",
    "has_conflict": false,
    "conflict_note": null
  },
  {
    "date": "2024-06-01",
    "staff_id": 2,
    "staff_name": "BS.B",
    "shift_type": "L03",
    "has_conflict": false,
    "conflict_note": null
  }
]


In [190]:
print("\n" + "=" * 70)
print("KET LUAN - RANKING")
print("=" * 70)


KET LUAN - RANKING


In [191]:
sorted_by_total = sorted(results, key=lambda x: x['total'], reverse=True)
print(f"\n{'Rank':<6} {'Algorithm':<22} {'TOTAL':<8} {'Time':<10} {'Fairness':<10} {'Coverage':<10}")
print("-" * 70)


Rank   Algorithm              TOTAL    Time       Fairness   Coverage  
----------------------------------------------------------------------


In [192]:
for i, r in enumerate(sorted_by_total, 1):
    print(f"{i:<6} {r['name']:<22} {r['total']:<8.2f} {r['time_ms']:<10.1f}ms {r['fairness']:<10.2f} {r['coverage']:<10.2f}")

1      Genetic                502.83   8795.7    ms 89.42      100.00    
2      Genetic                502.83   8795.7    ms 89.42      100.00    
3      Hybrid                 62.23    5033.1    ms 75.00      100.00    
4      Hybrid                 62.23    5033.1    ms 75.00      100.00    
5      Enhanced Greedy        61.70    2.0       ms 83.33      100.00    
6      Enhanced Greedy        61.70    2.0       ms 83.33      100.00    
7      Beam Search            49.86    9999.0    ms 91.60      99.17     
8      Beam Search            49.86    9999.0    ms 91.60      99.17     
9      Hill Climbing          49.34    2711.7    ms 91.60      99.17     
10     Hill Climbing          49.34    2711.7    ms 91.60      99.17     
11     Random Restart HC      35.45    1008.3    ms 91.53      98.33     
12     Random Restart HC      35.45    1008.3    ms 91.53      98.33     
13     Simulated Annealing    21.97    9929.8    ms 91.45      97.50     
14     Simulated Annealing    21.97   

In [193]:
print("\n" + "=" * 70)

In [194]:
print("\n=== TOP THEO TUNG METRIC ===")


=== TOP THEO TUNG METRIC ===


In [195]:
for metric, label in [('total', 'TOTAL'), ('fairness', 'FAIRNESS'),
                       ('fatigue', 'FATIGUE'), ('coverage', 'COVERAGE')]:
    top3 = sorted(results, key=lambda x: x[metric], reverse=True)[:3]
    print(f"\n  {label}:")
    for i, r in enumerate(top3, 1):
        print(f"    {i}. {r['name']} ({r[metric]:.2f})")


  TOTAL:
    1. Genetic (502.83)
    2. Genetic (502.83)
    3. Hybrid (62.23)

  FAIRNESS:
    1. Beam Search (91.60)
    2. Hill Climbing (91.60)
    3. Beam Search (91.60)

  FATIGUE:
    1. Beam Search (86.00)
    2. Beam Search (86.00)
    3. Hybrid (74.00)

  COVERAGE:
    1. Enhanced Greedy (100.00)
    2. Genetic (100.00)
    3. Hybrid (100.00)


In [196]:
print("\n" + "=" * 70)
print("NOTE: Tat ca scoring deu nam trong [0, 100], tong diem khong vuot 100.")
print("=" * 70)


NOTE: Tat ca scoring deu nam trong [0, 100], tong diem khong vuot 100.
